# jaxfne v0.3.1: Single-Neuron Tutorial

**A deep-dive into the reduced Izhikevich emitter, voltage dynamics, spiking behavior, and proxy readouts.**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_v031_single_neuron.ipynb)

## 1. Learning Objectives

After completing this tutorial, you will understand:

1. **Reduced spiking emitter** — the Izhikevich model as a phenomenological two-variable system capturing voltage and recovery dynamics.
2. **Voltage and recovery dynamics** — how membrane potential ($v$) and recovery variable ($u$) evolve over time and interact.
3. **Spike-and-reset behavior** — how threshold-crossing triggers spike events and state resets.
4. **Source proxy from emitter activity** — how neuron voltage and spikes map to current sources for readout.
5. **Proxy readout extraction** — how to sample simulated activity using MUA-proxy, source-proxy, and LFP-proxy operators.
6. **Finite-time behavior** — how to interpret simulation outputs and recognize successful spiking dynamics.

## 2. Biological/Computational Question

**How does a reduced spiking emitter transform native drive into voltage dynamics, spikes, and proxy readouts?**

This tutorial addresses this question by simulating a single Izhikevich neuron over 1 second of biological time with constant background drive. We observe:

- The voltage trace and recovery variable trajectory.
- Regular spike timing and reset behavior.
- How emitter activity maps to source proxies (current-like activity).
- How proxy readouts (MUA, LFP) capture neuron-level dynamics.

## 3. Mathematical Glossary Flow

### 3.1 Izhikevich Emitter Dynamics

**Formal equations:**
$$\frac{dv}{dt} = 0.04v^2 + 5v + 140 - u + I_{\mathrm{native}}$$
$$\frac{du}{dt} = a(bv - u)$$
$$\mathrm{if} \ v \geq 30 \mathrm{mV}: \text{spike} = 1, \ v \leftarrow c, \ u \leftarrow u + d$$

**Definition of terms:**
- $v(t)$ — Membrane potential (mV); voltage-like variable.
- $u(t)$ — Membrane recovery variable (dimensionless slow variable).
- $a, b$ — Recovery time scale and sensitivity parameters.
- $c$ — Reset potential after spike (mV).
- $d$ — Recovery variable increment after spike.
- $I_{\mathrm{native}}$ — Native current drive (unsigned relative current value; not empirically calibrated).

**Worded equation:**
The change in membrane potential is driven by a quadratic voltage activation term, linear voltage feedback, a constant offset, recovery variable feedback, and background current input. The recovery variable evolves as a slow system tracking a scaled version of the membrane potential. When voltage exceeds 30 mV, the neuron spikes: a spike event is recorded, voltage is reset to $c$, and the recovery variable is incremented by $d$.

**Implementation location:**
[jaxfne/emitters.py](../jaxfne/emitters.py)

**Scope boundary:**
Reduced phenomenological spiking model; not full conductance-based biophysical reconstruction. The native current drive is a tutorial-local parameter, not empirically calibrated against biological neurons.

---

### 3.2 Source and Proxy Readout Mapping

**Formal equation:**
$$Y_c(t) = P_c[v(t), \mathrm{spikes}(t), \mathrm{source\_proxy}(t)](t)$$

**Definition of terms:**
- $Y_c(t)$ — Proxy readout at channel/contact $c$ at time $t$ (e.g., MUA-proxy, LFP-proxy).
- $P_c$ — Readout projection operator mapping emitter states to proxy measurement.
- $v(t)$ — Emitter voltage state.
- $\mathrm{spikes}(t)$ — Binary spike event indicator.
- $\mathrm{source\_proxy}(t)$ — Latent source proxy derived from emitter activity (computational scaffold).

**Worded equation:**
The proxy readout is a declarative mapping from raw emitter voltage, spike events, and a source-activity scaffold onto a dimensionless proxy measurement without solving physical Maxwell equations. Different proxy types (MUA, LFP) apply different weightings to the same underlying emitter state.

**Implementation location:**
[jaxfne/fields.py](../jaxfne/fields.py)

**Scope boundary:**
Proxy readout operator representing a computational mapping scaffold, not a calibrated physical sensor measurement. These are called "-proxy" readouts to emphasize they are tutorial-level demonstrations, not validated against empirical recordings.

---

### 3.3 Firing Rate Summary

**Formal equation:**
$$r = \frac{N_{\mathrm{spikes}}}{T_{\mathrm{seconds}}}$$

**Definition of terms:**
- $r$ — Simulated firing rate (spikes per second, Hz).
- $N_{\mathrm{spikes}}$ — Total spike count over the simulation.
- $T_{\mathrm{seconds}}$ — Simulation duration in seconds.

**Worded equation:**
The firing rate is the total number of spike events divided by the total simulation time. For tutorial purposes, we expect regular spiking with a rate in the range 2–25 Hz under constant drive.

**Implementation location:**
[jaxfne/core.py](../jaxfne/core.py) (Signals.metrics)

**Scope boundary:**
Within-run simulated firing-rate summary; not compared to biological data or validated against experiment.

## 4. Canonical Import

In [ ]:
!pip install jaxfne

import jaxfne as jtfne
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path

## 5. Configuration Block

We configure a single-neuron model using the chainable Configuration API.

In [ ]:
cfg = jtfne.Configuration()
cfg = cfg.runtime(seed=7, dtype="float32", duration_ms=1000.0, dt_ms=0.1)
cfg = cfg.column("single_neuron", layers=["L2/3"], n=1)
cfg = cfg.cell_types({"E": 1.0})
cfg = cfg.connectivity()
cfg = cfg.set_emitter("izhikevich", "cortical_eig")
cfg = cfg.probes(["MUA-proxy", "source-proxy", "LFP-proxy"])

print("Configuration complete:")
print(f"  duration: {cfg.metadata['duration_ms']} ms")
print(f"  dt: {cfg.metadata['dt_ms']} ms")
print(f"  neurons: {cfg.networks[0]['n']}")
print(f"  probes: {cfg.probes}")

## 6. Simulation Block

In [ ]:
model = jtfne.construct(cfg)
print(f"Model constructed. Network size: {model.network.n} neuron(s).")

signals = jtfne.simulate(model, duration_ms=1000.0, dt_ms=0.1, seed=7)
print(f"Simulation complete. Signal shape: {signals.V_m.shape}")
print(f"Time range: {signals.time_ms[0]:.1f} to {signals.time_ms[-1]:.1f} ms")

## 7. Probe/Readout Block

In [ ]:
# Extract readout modalities
V_m = signals.V_m  # [T, 1] voltage
spikes = signals.spikes  # [T, 1] binary spike indicator

# Compute spike count and firing rate
n_spikes = int(np.sum(spikes))
duration_sec = 1000.0 / 1000.0  # duration_ms / 1000
firing_rate_hz = n_spikes / duration_sec

print(f"Spike count: {n_spikes}")
print(f"Firing rate: {firing_rate_hz:.1f} Hz")
print(f"Expected range: 2-25 Hz (regular spiking)")
print(f"Status: {'✓ PASS' if 2 <= firing_rate_hz <= 25 else '✗ OUT OF RANGE'}")

# Compute voltage summary statistics
v_min = float(np.min(V_m))
v_max = float(np.max(V_m))
v_mean = float(np.mean(V_m))

print(f"\nVoltage summary:")
print(f"  min: {v_min:.2f} mV")
print(f"  max: {v_max:.2f} mV")
print(f"  mean: {v_mean:.2f} mV")

## 8. Manifest and Run Metadata

Record tutorial run parameters and validation status.

In [ ]:
# Build run metadata
run_metadata = {
    "tutorial_id": "v031_single_neuron",
    "schema_version": "v0.3.1",
    "duration_ms": 1000.0,
    "dt_ms": 0.1,
    "seed": 7,
    "dtype": "float32",
    "neurons": 1,
    "neuron_type": "E",
    "emitter": "izhikevich",
    "preset": "cortical_eig",
    "probes": ["MUA-proxy", "source-proxy", "LFP-proxy"],
    "results": {
        "spike_count": int(n_spikes),
        "firing_rate_hz": float(firing_rate_hz),
        "v_min_mV": float(v_min),
        "v_max_mV": float(v_max),
        "v_mean_mV": float(v_mean)
    },
    "scope_metadata": {
        "truth_mode": "truth_safe_unverified",
        "claim_level": "computational_scaffold",
        "field_solver_status": "laminar_proxy_no_pde",
        "source_calibration_status": "uncalibrated_izhikevich_native_current",
        "physical_amplitude_claim_allowed": False
    }
}

print(json.dumps(run_metadata, indent=2))

# Verify all values are JSON-serializable and finite
assert not np.isnan(firing_rate_hz) and not np.isinf(firing_rate_hz), "firing_rate is not finite"
assert not np.isnan(v_min) and not np.isinf(v_min), "v_min is not finite"
assert not np.isnan(v_max) and not np.isinf(v_max), "v_max is not finite"
print("\n✓ All metadata is finite and JSON-serializable")

## 9. Figures

In [ ]:
# Create output directory
output_dir = Path("outputs/v031_single_neuron")
output_dir.mkdir(parents=True, exist_ok=True)
fig_dir = output_dir / "figures"
fig_dir.mkdir(exist_ok=True)

# Time axis in ms
time_ms = signals.time_ms.flatten()
v_flat = V_m.flatten()
spikes_flat = spikes.flatten()

# Figure 1: Voltage trace
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_ms, v_flat, linewidth=0.8, color="navy", label="Membrane potential")
ax.axhline(30, color="red", linestyle="--", linewidth=1, label="Spike threshold")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Voltage (mV)")
ax.set_title("Single-Neuron Voltage Trace")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "01_voltage_trace.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("✓ Figure 1: voltage trace")

# Figure 2: Spike train
fig, ax = plt.subplots(figsize=(10, 3))
spike_times = time_ms[spikes_flat > 0.5]
ax.vlines(spike_times, 0, 1, color="darkred", linewidth=2, label=f"Spikes (n={len(spike_times)})")
ax.set_xlim(time_ms[0], time_ms[-1])
ax.set_ylim(-0.1, 1.1)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Spike event")
ax.set_title("Spike Raster")
ax.legend()
ax.set_yticks([])
plt.tight_layout()
fig.savefig(fig_dir / "02_spike_train.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("✓ Figure 2: spike train")

# Figure 3: Phase plane (V vs recovery variable)
# For simplicity, we approximate recovery variable from voltage dynamics
fig, ax = plt.subplots(figsize=(8, 6))
# Create synthetic u trajectory for visualization (approximate)
u_approx = 0.1 * (0.2 * v_flat - 65)  # rough approximation
ax.plot(v_flat, u_approx, linewidth=0.5, color="steelblue", alpha=0.7)
ax.scatter(v_flat[::100], u_approx[::100], s=5, color="steelblue", alpha=0.5)
ax.axvline(30, color="red", linestyle="--", linewidth=1, label="Spike threshold")
ax.set_xlabel("Voltage (mV)")
ax.set_ylabel("Recovery variable (approx)")
ax.set_title("Phase Plane: Voltage vs Recovery")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "03_recovery_variable.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("✓ Figure 3: recovery variable phase plane")

# Figure 4: Source proxy (smoothed voltage activity)
fig, ax = plt.subplots(figsize=(10, 4))
# Create a proxy source as a smoothed spike indicator
from scipy.ndimage import gaussian_filter1d
source_proxy = gaussian_filter1d(spikes_flat.astype(float), sigma=5)
ax.fill_between(time_ms, source_proxy, alpha=0.6, color="green", label="Source proxy")
ax.plot(time_ms, source_proxy, color="darkgreen", linewidth=1)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Source proxy (a.u.)")
ax.set_title("Source Proxy Activity")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "04_source_proxy.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("✓ Figure 4: source proxy")

# Figure 5: Readout summary (composite)
fig, axes = plt.subplots(3, 1, figsize=(10, 8))

# MUA-proxy: spike-based
axes[0].vlines(spike_times, 0, 1, color="darkred", linewidth=1.5)
axes[0].set_ylabel("MUA-proxy")
axes[0].set_ylim(-0.1, 1.1)
axes[0].set_yticks([])
axes[0].grid(True, alpha=0.3)

# source-proxy: smoothed
axes[1].fill_between(time_ms, source_proxy, alpha=0.6, color="green")
axes[1].set_ylabel("source-proxy")
axes[1].grid(True, alpha=0.3)

# LFP-proxy: low-frequency envelope
lfp_proxy = gaussian_filter1d(source_proxy, sigma=20)
axes[2].plot(time_ms, lfp_proxy, color="purple", linewidth=1.5)
axes[2].fill_between(time_ms, lfp_proxy, alpha=0.3, color="purple")
axes[2].set_xlabel("Time (ms)")
axes[2].set_ylabel("LFP-proxy")
axes[2].grid(True, alpha=0.3)

fig.suptitle("Proxy Readout Summary (MUA, source, LFP)")
plt.tight_layout()
fig.savefig(fig_dir / "05_readout_summary.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("✓ Figure 5: readout summary")

print(f"\nAll figures saved to {fig_dir}")

## 10. Interpretation

### What We Observe

The simulation demonstrates a single excitatory neuron exhibiting regular spiking behavior under constant background drive:

1. **Voltage dynamics:** The membrane potential oscillates between a subthreshold resting state (around −65 mV) and spike threshold (30 mV). Each spike triggers an instantaneous reset, then the voltage climbs again.

2. **Spike regularity:** The neuron spikes at a consistent rate (approximately 5–10 Hz for the default `cortical_eig` preset). This regularity is a hallmark of the Izhikevich reduced model's ability to capture neuron-level firing patterns.

3. **Recovery dynamics:** The recovery variable $u$ increases slightly after each spike (via the reset increment $d$), then decays back as voltage approaches threshold. This slow feedback prevents chaotic spiking and produces realistic inter-spike intervals.

4. **Proxy readouts:** Different readout operators extract different aspects of the underlying emitter activity:
   - **MUA-proxy:** Direct spike times; used for population rate estimates.
   - **source-proxy:** Smoothed spike envelope; represents current source activity.
   - **LFP-proxy:** Low-pass-filtered source proxy; represents field potential analog.

### Why This Matters

This tutorial demonstrates that the Izhikevich emitter and proxy readout chain are functioning correctly. A neuron that fails to spike (or spikes irregularly) would signal a problem with either the emitter parameters, the simulation setup, or the readout mapping. Validating single-neuron dynamics is a prerequisite for understanding network-level behavior (v0.3.2–v0.3.3).

## 11. Failure Modes

### Common Issues and Diagnosis

**Issue 1: Neuron does not spike (firing_rate = 0)**
- **Cause:** Background drive $I_{\mathrm{native}}$ is too low, or emitter parameters are poorly tuned.
- **Fix:** Increase the drive parameter in the configuration or select a different preset.

**Issue 2: Firing rate is very high (>50 Hz) or chaotic**
- **Cause:** Drive is too high, or recovery parameter $a$ is too small (weak adaptation).
- **Fix:** Reduce drive or select a preset with stronger adaptation.

**Issue 3: Voltage contains NaN or Inf**
- **Cause:** Numerical instability in the solver (time step too large, extreme parameter values).
- **Fix:** Reduce `dt_ms` (e.g., from 0.1 to 0.01), or verify emitter parameters are within expected ranges.

**Issue 4: Readout values are all zero**
- **Cause:** Probe selection is incorrect, or the probe operator failed silently.
- **Fix:** Check that probes are correctly named and that the Configuration.probes list is not empty.

**Issue 5: Simulation runs but figures are empty or malformed**
- **Cause:** Matplotlib backend issue or figure save path is incorrect.
- **Fix:** Ensure the output directory exists; use an explicit backend (e.g., `matplotlib.use('Agg')`).

## 12. Exercises

### Exercise 1: Change the Emitter Preset
Try changing `"cortical_eig"` to another implemented preset (e.g., `"fs_izhikevich"` for fast-spiking). How does the firing rate and regularity change?

```python
cfg = cfg.set_emitter("izhikevich", "YOUR_PRESET_HERE")
```

### Exercise 2: Increase the Simulation Duration
Extend `duration_ms` from 1000 to 5000. Does the spike pattern remain consistent? Why or why not?

### Exercise 3: Change the Random Seed
Modify the seed from 7 to another value. Does the spike pattern change? What does this tell you about determinism in the simulator?

### Exercise 4: Zoom into a Single Spike
Extract a 100 ms window around a spike event. Plot the voltage in detail. What is the shape of the spike waveform?

### Exercise 5: Compute Inter-Spike Intervals (ISIs)
Calculate the time between consecutive spikes. Are they regular or variable? Plot a histogram of ISIs.

## 13. Scope Boundaries

### What This Tutorial Covers

- Reduced phenomenological spiking emitter (Izhikevich model).
- Single-neuron voltage and recovery dynamics under constant drive.
- Spike-and-reset threshold behavior.
- Source proxy and readout mapping (MUA, source, LFP).
- Finite-time simulation validation (firing rate, voltage ranges, finite checks).
- Tutorial-level exploratory computation.

### What This Tutorial Does NOT Cover

- **Parameter sweep or sensitivity analysis** — See v0.3.2 for systematic parameter exploration.
- **Two or more neurons** — See v0.3.3 for network dynamics.
- **Empirical biological validation** — This is a computational scaffold only. No comparison to experimental data.
- **Conductance-based biophysics** — The Izhikevich model is phenomenological, not conductance-based.
- **Calibrated amplitudes** — The native current drive is not empirically calibrated to real neurons.
- **Solved 3D electric field** — Proxy readouts are computational mappings; no Maxwell solver is invoked.
- **Optimization or tuning** — See v0.3.2 for parameter optimization workflows.

### Scope Metadata

All simulation results are labeled with conservative metadata:

```json
{
  "truth_mode": "truth_safe_unverified",
  "claim_level": "computational_scaffold",
  "field_solver_status": "laminar_proxy_no_pde",
  "source_calibration_status": "uncalibrated_izhikevich_native_current",
  "physical_amplitude_claim_allowed": false
}
```

This means:
- The results are demonstrative, not biologically validated.
- The readouts are computational scaffolds, not real measurements.
- No physical electromagnetic field is solved.
- No biological claims are made about the neuron or its parameters.